# 21_model_rule_baseline_CHS

- 목적: 규칙 기반 베이스라인 4종(크기 · 상태별 크기 · 크기×방향)을 공통 평가 함수로 비교하고, 갭 검증 3개(V2-a FN 19·20 전류 이상의 DC/AC, V2-b 운전 상태 정의, V2-c 크기×방향 효과)를 수치로 확인한다.
- 입력: `data/processed/11_window_features.parquet`, `data/processed/segments.csv` (원시 데이터 로드 없음)
- 출력: `figures/21_model_rule_baseline_CHS/`, `results/21_model_rule_baseline_CHS/` (`metrics.csv`, `fn_channel_table.csv`, `state_definition.csv`, `state_baseline.csv`, `error_cases.csv`)
- 담당: CHS
- 심사 대응: 2번(베이스라인 비교), 3번(FN 조건), 5번(방향·상태 결합 규칙)

- 목적: 
- 입력: `data/processed/…` 또는 이전 노트북
- 출력: `figures/<노트북명>/`, `results/<노트북명>/`, `data/processed/<번호>_…`
- 담당: <이니셜>

파일명 규칙: 0x 진단 · 1x 전처리·피처 · 2x 모델 · 3x 분석, 예: 21_model_iforest_CHS.ipynb — 복사 후 NB 값을 파일명과 같게 바꾼다

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "21_model_rule_baseline_CHS"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
import evaluate, split
from sklearn.cluster import KMeans
PQ = paths.DATA_PROCESSED / "11_window_features.parquet"
SEG = paths.DATA_PROCESSED / "segments.csv"
print(NB, FIG.relative_to(ROOT), RES.relative_to(ROOT))


21_model_rule_baseline_CHS figures\21_model_rule_baseline_CHS results\21_model_rule_baseline_CHS


## 1. 규칙 정의와 공통 함수

점수는 클수록 이상, 임계 = train fold **정상** 점수 q99(`evaluate.threshold_from_normal`). 이상 데이터·`label`·`src`·절대 시간은 규칙 입력에 쓰지 않는다(`label`은 평가에만).

| model | 점수 | 비고 |
|---|---|---|
| `rule1_rms_ai0` | `AI0_Vibration_rms` | 크기 1차원. 11 노트북 smoke와 같아야 함 |
| `rule2_rms_state` | `AI0_Vibration_rms / thr_state` | 상태별 크기. 상태 = 윈도우 자신의 DC 제거 후 전류 RMS(`AI2_Current_rms`)를 train 정상 윈도우 2-means 중점 경계로 저/고 판정, `thr_state` = train 정상의 상태별 AI0 rms q99 |
| `rule3_rms_state_dir` | `(AI0_Vibration_rms / thr_state) × (1 − vib_corr01)` | 크기×방향. `vib_corr01`은 **세그먼트 브로드캐스트 값**(nan은 0). 상관이 음수일수록 이상 방향 |
| `rule3b_rms_state_ratio` | `(AI0_Vibration_rms / thr_state) × vib_rms_ratio` | 방향 지표를 윈도우 단위 AI0/AI1 rms 비율로 대체(nan은 train 정상 중앙값) |

상태 배정은 테스트 윈도우(이상 포함)도 자기 값으로 한다. 상태 0 = 저부하, 1 = 고부하(2-means 중심 크기순).

In [2]:
W_ALL = pd.read_parquet(PQ)
S = pd.read_csv(SEG, encoding="utf-8-sig")
MODELS = ["rule1_rms_ai0", "rule2_rms_state", "rule3_rms_state_dir", "rule3b_rms_state_ratio"]
CUR, VIB = "AI2_Current_rms", "AI0_Vibration_rms"
EXTRA = ["thr", "n_out_seg", "n_detected", "felt_delay_median_s"]

def fit_state(cur_train_normal):
    """train 정상 윈도우 전류 AC RMS 2-means -> (경계=두 중심의 중점, 저/고 중심)."""
    km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(np.asarray(cur_train_normal, float).reshape(-1, 1))
    lo, hi = np.sort(km.cluster_centers_.ravel())
    return (lo + hi) / 2, (lo, hi)

def rule_scores(w, tr_mask):
    """w 전체 행에 대한 규칙 4종 점수(DataFrame)와 fit 정보. 모든 fit은 tr_mask & 정상 윈도우만 사용."""
    trn = w[tr_mask & (w.label == 0)]
    bound, _ = fit_state(trn[CUR])
    st = (w[CUR] > bound).astype(int).to_numpy()
    st_tr = (trn[CUR] > bound).astype(int).to_numpy()
    thr_state = {s: evaluate.threshold_from_normal(trn[VIB].to_numpy()[st_tr == s]) for s in (0, 1)}
    rel = w[VIB].to_numpy() / np.where(st == 1, thr_state[1], thr_state[0])
    ratio_fill = float(np.nanmedian(trn["vib_rms_ratio"]))
    sc = pd.DataFrame({
        "rule1_rms_ai0": w[VIB].to_numpy(),
        "rule2_rms_state": rel,
        "rule3_rms_state_dir": rel * (1.0 - w["vib_corr01"].fillna(0.0).to_numpy()),
        "rule3b_rms_state_ratio": rel * w["vib_rms_ratio"].fillna(ratio_fill).to_numpy(),
    }, index=w.index)
    return sc, {"state_boundary": bound, "thr_state_low": thr_state[0], "thr_state_high": thr_state[1], "state": st}

def eval_split(w, sc, tr_mask, te_idx, split_name, fold_id, win_s):
    """규칙 4종 한 번에 평가. 반환: (metrics 행 리스트, {model: 이상 세그먼트별 탐지 표})."""
    rows, dets = [], {}
    for m in MODELS:
        s = sc[m]
        thr = evaluate.threshold_from_normal(s[tr_mask & (w.label == 0)])
        te = w.loc[te_idx].assign(_s=s.loc[te_idx]); n, o = te[te.label == 0], te[te.label == 1]
        dl = evaluate.detection_delay_s(o._s, o.seg_uid, o.t_start, thr, win_s=win_s)
        dg = evaluate.detection_delay_s(o._s, o.seg_uid, o.t_start, thr, win_s=win_s, include_gap=True)
        r = evaluate.to_metrics_row(m, split_name, fold_id, evaluate.auc(te.label, te._s),
                                    evaluate.fpr_sample(n._s, thr), evaluate.fpr_segment(n._s, n.seg_uid, thr),
                                    dl["delay_s"].median())
        r.update({"win_s": win_s, "thr": thr, "n_out_seg": len(dl), "n_detected": int(dl["detected"].sum()),
                  "felt_delay_median_s": dg["delay_s"].median()})
        rows.append(r); dets[m] = dl
    return rows, dets
print(W_ALL.groupby(["win_s", "label"]).size().unstack())

label     0   1
win_s          
1.0    3264  96
2.0    2254  63
3.0    1400  38


## 2. V2-a — FN 19·20의 전류 이상은 DC인가 AC인가

이상 세그먼트 21개에 대해 `segments.csv`의 전류 RMS·DC(원시, DC 포함)와 parquet의 DC 제거 후 1초 윈도우 전류 RMS(AC) 중앙값, 세그먼트 사인 피팅 R²(`cur_fit_r2`), `vib_corr01`, AI0 rms를 나란히 놓는다. 1초 윈도우가 없는 세그먼트(길이 < 1초 = 10샘플)는 NaN.

In [3]:
W1 = W_ALL[W_ALL.win_s == 1.0].reset_index(drop=True)
out_seg = S[S.label == 1].copy()
g = W1[W1.label == 1].groupby("seg_uid").agg(cur_ac_rms_1s_median=(CUR, "median"), cur_fit_r2=("cur_fit_r2", "first"),
                                             vib_corr01=("vib_corr01", "first"), ai0_rms_1s_median=(VIB, "median"))
FN = (out_seg.merge(g, left_on="seg_uid", right_index=True, how="left")
      .rename(columns={"AI2_Current_rms": "seg_cur_rms_raw", "AI2_Current_dc": "seg_cur_dc"}))
FN["seg_id"] = FN["seg_id"].astype(int)
FN["dc_energy_share"] = FN["seg_cur_dc"] ** 2 / FN["seg_cur_rms_raw"] ** 2   # DC가 RMS^2에서 차지하는 비중
FN = FN[["seg_uid", "seg_id", "len", "seg_cur_rms_raw", "seg_cur_dc", "dc_energy_share", "cur_ac_rms_1s_median",
         "cur_fit_r2", "vib_corr01", "ai0_rms_1s_median", "vib_grade", "cur_grade"]]
FN.to_csv(RES / "fn_channel_table.csv", index=False)
print(FN.round(3).to_string(index=False))
nrm = W1[W1.label == 0]
print("\n정상 1초 윈도우: 전류 AC RMS 중앙값 %.1f (q1 %.1f, q99 %.1f), cur_fit_r2 중앙값 %.3f (q1 %.3f)" % (
    nrm[CUR].median(), nrm[CUR].quantile(.01), nrm[CUR].quantile(.99), nrm.cur_fit_r2.median(), nrm.cur_fit_r2.quantile(.01)))
seg_n = S[S.label == 0]
print("정상 세그먼트: 전류 |DC| 중앙값 %.1f, 원시 RMS 중앙값 %.1f" % (seg_n.AI2_Current_dc.abs().median(), seg_n.AI2_Current_rms.median()))
for i in (19, 20):
    r = FN[FN.seg_id == i].iloc[0]
    print("outlier_%d: 원시 RMS %.1f, DC %.1f (RMS^2 중 %.1f%%), AC RMS(1초 중앙값) %.1f, R2 %.3f" % (
        i, r.seg_cur_rms_raw, r.seg_cur_dc, 100 * r.dc_energy_share, r.cur_ac_rms_1s_median, r.cur_fit_r2))

   seg_uid  seg_id  len  seg_cur_rms_raw  seg_cur_dc  dc_energy_share  cur_ac_rms_1s_median  cur_fit_r2  vib_corr01  ai0_rms_1s_median vib_grade cur_grade
 outlier_0       0    4           57.663     -24.736            0.184                   NaN         NaN         NaN                NaN     확실 이상        경계
 outlier_1       1   50           99.049      43.988            0.197                92.229       0.309      -0.436              0.421     확실 이상        경계
 outlier_2       2   47          102.031      -9.258            0.008               100.924       0.641      -0.357              0.621     확실 이상        경계
 outlier_3       3   31           99.127      73.448            0.549                73.791       0.291       0.184              0.029     정상 유사        경계
 outlier_4       4   18          153.573    -120.269            0.613                93.927       0.451      -0.348              0.732     확실 이상        경계
 outlier_5       5    3          143.096    -141.859            0.983 

### V2-a 결론 — 19·20의 전류 "이상"은 DC 오프셋이지 AC 진폭 증가가 아니다

- `fn_channel_table.csv` 기준 19·20의 원시 전류 RMS 207.2 / 183.4 중 **DC가 198.5 / 180.1**로 RMS²의 **91.7% / 96.4%**를 차지한다. DC 제거 후 1초 윈도우 전류 RMS(AC) 중앙값은 **59.6 / 35.2**로, 정상 윈도우 중앙값 100.5(q1 68.9)보다 오히려 낮다(20은 정상 1% 분위수 아래). 정상 세그먼트 |DC| 중앙값은 4.9.
- 따라서 표준 전처리(세그먼트 평균 제거) 이후에는 19·20의 전류 이상이 **크기 증가로 보이지 않는다**. `cur_grade`의 "확실 이상"은 DC(형식 누수 의심 채널) 값에 기대고 있어, "전류 단독으로 확실한 이상"이라는 해석은 그대로 쓰면 안 된다.
- AC 쪽 근거인 사인 R²는 19·20이 0.464 / 0.349로 정상(중앙값 0.996, q1 0.994)보다 크게 낮다. 그러나 1초 윈도우가 있는 이상 세그먼트 17개 모두 R² 0.29~0.78로 정상 범위 밖이라 19·20만의 특징이 아니며, 세그먼트 수준·날짜 교란 가능성이 있는 값이다.
- 시사점(가설): 20은 AC 진폭이 정상 하한 아래로 내려간 저전류 쪽 이상일 수 있다. q99 단측 규칙으로는 잡히지 않으므로 전류 AC RMS의 **하한 임계(양측 검정)** 는 후속 모델 노트북의 검토 항목이다. 이상 이벤트가 1건이라 일반화 근거는 아니다.
- 1초 윈도우가 없는 세그먼트 0·5·9·10(4개)은 AC·R² 값이 NaN이며, 원시 DC 비중만 확인 가능하다(0.184 / 0.983 / 0.819 / 0.861).

## 3. V2-b — 운전 상태 정의

(1) 문서 임계(세그먼트 전류 RMS < 95 / 95~140 / > 140) × `state`(k-means, 정상 599 세그먼트) 교차표. (2) 입력 가능한 값인 DC 제거 후 윈도우 전류 RMS로 상태를 나눌 수 있는지 — 정상 1초 윈도우 2-means 경계와 세그먼트 정의와의 일치율. (3) 두 상태 정의 각각에 대한 상태별 기준선(AI0 rms, `vib_corr01`, `vib_rms_ratio`).

In [4]:
seg_n = S[S.label == 0].copy()
seg_n["cur_bin"] = pd.cut(seg_n.AI2_Current_rms, [-np.inf, 95, 140, np.inf], labels=["<95", "95~140", ">140"])
xt = pd.crosstab(seg_n.cur_bin, seg_n.state.astype(int), margins=True)
xt.columns = [f"state{c}" if c != "All" else "All" for c in xt.columns]
xt.to_csv(RES / "state_definition.csv"); print(xt)
print("state별 세그먼트 전류 RMS 중앙값:", seg_n.groupby("state").AI2_Current_rms.median().round(1).to_dict(), "(state 0 = 고부하, 1 = 저부하로 확인)")

# 윈도우 AC RMS 2-means (정상 1초 윈도우 전체로 정의용 fit)
wn = W1[W1.label == 0].copy()
bound_all, (c_lo, c_hi) = fit_state(wn[CUR])
wn["win_high"] = (wn[CUR] > bound_all).astype(int)
# parquet state(세그먼트 k-means): 0 = 고부하 -> win_high와 맞추기 위해 뒤집는다
wn["seg_high"] = (wn.state == 0).astype(int)
agree_kmeans = float((wn.win_high == wn.seg_high).mean())
wn = wn.merge(seg_n[["seg_uid", "AI2_Current_rms", "cur_bin"]].rename(columns={"AI2_Current_rms": "seg_cur_rms"}), on="seg_uid", how="left")
clear = wn[wn.cur_bin.isin(["<95", ">140"])]
agree_clear = float((clear.win_high == (clear.cur_bin == ">140").astype(int)).mean())
mid = wn[wn.cur_bin == "95~140"]
bd = pd.DataFrame([{"win_2means_boundary": bound_all, "center_low": c_lo, "center_high": c_hi,
                    "n_windows": len(wn), "agree_with_segment_kmeans_state": agree_kmeans,
                    "agree_with_seg_threshold_clear(<95,>140)": agree_clear, "n_clear_windows": len(clear),
                    "mid95_140_windows": len(mid), "mid95_140_share_high": float(mid.win_high.mean())}])
bd.to_csv(RES / "state_window_boundary.csv", index=False)
print(bd.T.round(4).to_string(header=False))
print(pd.crosstab(wn.cur_bin, wn.win_high, margins=True))

rows = []
for defn, col in [("segment_kmeans", "seg_high"), ("window_2means", "win_high")]:
    for s_, name in [(0, "low"), (1, "high")]:
        sub = wn[wn[col] == s_]
        for f in [VIB, "vib_corr01", "vib_rms_ratio"]:
            rows.append({"definition": defn, "state": name, "feature": f, "n_windows": len(sub),
                         "median": sub[f].median(), "q99": sub[f].quantile(.99)})
SB = pd.DataFrame(rows); SB.to_csv(RES / "state_baseline.csv", index=False)
print(SB.round(4).to_string(index=False))

         state0  state1  All
cur_bin                     
<95           0     289  289
95~140       64      58  122
>140        188       0  188
All         252     347  599
state별 세그먼트 전류 RMS 중앙값: {0.0: 159.6, 1.0: 85.1} (state 0 = 고부하, 1 = 저부하로 확인)


win_2means_boundary                        125.2054
center_low                                  90.1978
center_high                                160.2129
n_windows                                 3264.0000
agree_with_segment_kmeans_state              0.9442
agree_with_seg_threshold_clear(<95,>140)     0.9953
n_clear_windows                           2566.0000
mid95_140_windows                          698.0000
mid95_140_share_high                         0.3238


win_high     0     1   All
cur_bin                   
<95       1528     0  1528
95~140     472   226   698
>140        12  1026  1038
All       2012  1252  3264
    definition state           feature  n_windows  median    q99
segment_kmeans   low AI0_Vibration_rms       1842  0.0508 0.0992
segment_kmeans   low        vib_corr01       1842  0.1120 0.5830
segment_kmeans   low     vib_rms_ratio       1842  0.9114 2.2546
segment_kmeans  high AI0_Vibration_rms       1422  0.0800 0.1384
segment_kmeans  high        vib_corr01       1422  0.4546 0.6802
segment_kmeans  high     vib_rms_ratio       1422  0.4957 1.0453
 window_2means   low AI0_Vibration_rms       2012  0.0529 0.1116
 window_2means   low        vib_corr01       2012  0.1278 0.5818
 window_2means   low     vib_rms_ratio       2012  0.8885 2.2426
 window_2means  high AI0_Vibration_rms       1252  0.0792 0.1373
 window_2means  high        vib_corr01       1252  0.4747 0.7307
 window_2means  high     vib_rms_ratio       1252  0.4776 

### V2-b 결론 — 상태는 입력 가능한 윈도우 AC RMS의 2-means로 정의할 수 있다

- 세그먼트 전류 RMS 임계와 `state`(k-means) 교차표(정상 599): `<95` 289개는 전부 state 1(저부하), `>140` 188개는 전부 state 0(고부하), `95~140` 122개는 state 0 64 / state 1 58로 섞인다. 즉 두 정의는 양 끝에서 일치하고 95~140 구간이 완충 구간이다. (parquet `state`는 0 = 고부하, 1 = 저부하. 위 규칙 셀의 상태 번호와 반대이므로 비교는 고/저 이름으로 한다.)
- 정상 1초 윈도우(3,264개) 전류 AC RMS 2-means: 중심 90.20 / 160.21, **경계 125.21**. 세그먼트 k-means 상태와의 일치율 **94.42%**, 임계가 명확한 구간(`<95`, `>140`, 2,566 윈도우)의 일치율 **99.53%**. 95~140 구간 698개 윈도우 중 32.38%가 고부하로 판정된다.
- 결론: 상태 판정에 원시 DC를 쓸 필요 없이 **DC 제거 후 윈도우 전류 RMS 하나로 2상태**를 나눌 수 있고(입력 가능), 3구간 임계(95/140)는 2상태 경계 하나(≈125)로 단순화해도 된다.
- 상태별 기준선(1초 정상 윈도우, 윈도우 2-means 정의): AI0 rms 중앙값/q99 — 저부하 0.0529 / 0.1116, 고부하 0.0792 / 0.1373. `vib_corr01` 중앙값 — 저 0.128, 고 0.475. `vib_rms_ratio` 중앙값 — 저 0.889, 고 0.478. 세 지표 모두 상태에 크게 의존하므로 방향 지표(상관·비율)를 쓸 때도 상태별 기준화가 필요하다.

## 4. V2-c — 규칙 4종 평가 (윈도우 1·2·3초 × 분할 2종)

`fold`(GroupKFold 5, seed 42; test = fold==k)와 `block`(train = block<k, test = block==k + `split.append_outliers`, k=1..4). 상태·임계는 매 분할마다 train 정상만으로 새로 fit한다. 지연은 윈도우 완성 시각 기준, 체감 지연은 `include_gap=True`.

In [5]:
all_rows, OOF, FITS = [], {}, []
for win_s in (1.0, 2.0, 3.0):
    w = W_ALL[W_ALL.win_s == win_s].reset_index(drop=True)
    for k in range(5):
        tr = (w.fold != k).to_numpy(); te = np.flatnonzero(w.fold == k)
        sc, info = rule_scores(w, tr)
        rows, dets = eval_split(w, sc, tr, te, "group_kfold_seg", k, win_s); all_rows += rows
        FITS.append({"win_s": win_s, "split": "group_kfold_seg", "fold": k, **{a: b for a, b in info.items() if a != "state"}})
        if win_s == 1.0:
            for m, d in dets.items(): OOF.setdefault(m, []).append(d)
    for k in range(1, 5):
        tr = (w.block < k).fillna(False).to_numpy(bool)
        te = split.append_outliers(np.flatnonzero((w.block == k).fillna(False).to_numpy(bool)), w)
        sc, info = rule_scores(w, tr)
        rows, _ = eval_split(w, sc, tr, te, "time_block", k, win_s); all_rows += rows
        FITS.append({"win_s": win_s, "split": "time_block", "fold": k, **{a: b for a, b in info.items() if a != "state"}})
M = pd.DataFrame(all_rows)
mean_rows = []
for (sp, m, ws), g_ in M.groupby(["split", "model", "win_s"], sort=False):
    r = g_.drop(columns=["model", "split", "fold"]).mean(numeric_only=True).to_dict()
    r.update({"model": m, "split": sp, "fold": "mean"}); mean_rows.append(r)
M = pd.concat([M, pd.DataFrame(mean_rows)], ignore_index=True); M["fold"] = M["fold"].astype(str)
M = M[evaluate.METRIC_COLS + ["win_s"] + EXTRA]
M.to_csv(RES / "metrics.csv", index=False)
pd.DataFrame(FITS).to_csv(RES / "state_fit_by_fold.csv", index=False)
show = M[M.fold == "mean"].copy()
print(show.round(4).to_string(index=False))

                 model           split fold    auc  fpr_sample  fpr_segment  delay_median_s  win_s    thr  n_out_seg  n_detected  felt_delay_median_s
         rule1_rms_ai0 group_kfold_seg mean 0.7864      0.0109       0.0395            0.95    1.0 0.1278        3.4         2.8                8.908
       rule2_rms_state group_kfold_seg mean 0.7996      0.0119       0.0468            0.95    1.0 1.0004        3.4         2.8                8.908
   rule3_rms_state_dir group_kfold_seg mean 0.8696      0.0138       0.0358            0.95    1.0 0.8196        3.4         2.8                8.908
rule3b_rms_state_ratio group_kfold_seg mean 0.8058      0.0105       0.0399            0.95    1.0 1.4731        3.4         2.8                8.908
         rule1_rms_ai0      time_block mean 0.7786      0.0116       0.0384            0.90    1.0 0.1268       17.0        14.0                8.858
       rule2_rms_state      time_block mean 0.7884      0.0139       0.0456            0.90    1.0 1

### 4.1 재현 확인 — rule1 1초는 11 노트북 smoke와 같아야 한다

In [6]:
sm = pd.read_csv(paths.RESULTS / "11_window_features_CHS" / "metrics_smoke.csv")
cols = ["auc", "fpr_sample", "fpr_segment", "delay_median_s", "thr", "n_out_seg", "n_detected", "felt_delay_median_s"]
mine = M[(M.model == "rule1_rms_ai0") & (M.win_s == 1.0)]
ok = True
for sp in ("group_kfold_seg", "time_block"):
    a = mine[mine.split == sp].set_index("fold")[cols].astype(float)
    b = sm[sm.split == sp].astype({"fold": str}).set_index("fold")[cols].astype(float)
    same = np.allclose(a.loc[b.index].to_numpy(), b.to_numpy(), equal_nan=True, rtol=1e-9, atol=1e-12)
    print(sp, "일치" if same else "불일치"); ok &= same
assert ok

group_kfold_seg 일치
time_block 일치


## 5. 오류 사례 — 1초 kfold OOF 탐지 여부 (이상 세그먼트 21개 × 규칙 4종)

1초 윈도우가 없는 세그먼트(길이 < 10샘플)는 `NaN`(판정 불가). 탐지 = 해당 fold의 train 정상 q99 임계를 넘는 윈도우가 1개 이상.

In [7]:
ec = FN[["seg_uid", "seg_id", "len", "vib_grade", "cur_grade", "seg_cur_rms_raw", "seg_cur_dc", "cur_ac_rms_1s_median",
         "cur_fit_r2", "vib_corr01", "ai0_rms_1s_median"]].copy()
for m in MODELS:
    d = pd.concat(OOF[m]).set_index("seg_uid")["detected"]
    ec[f"det_{m}"] = ec.seg_uid.map(d).astype(object)     # 윈도우 없으면 NaN
ec.to_csv(RES / "error_cases.csv", index=False)
print(ec.drop(columns=["seg_cur_rms_raw", "seg_cur_dc"]).round(3).to_string(index=False))
print("\n규칙별 탐지/평가가능 세그먼트:")
for m in MODELS:
    c = ec[f"det_{m}"].dropna().astype(bool); print(f"  {m}: {int(c.sum())}/{len(c)}  미탐 seg_id = {ec.loc[c.index[~c], 'seg_id'].tolist()}")

   seg_uid  seg_id  len vib_grade cur_grade  cur_ac_rms_1s_median  cur_fit_r2  vib_corr01  ai0_rms_1s_median det_rule1_rms_ai0 det_rule2_rms_state det_rule3_rms_state_dir det_rule3b_rms_state_ratio
 outlier_0       0    4     확실 이상        경계                   NaN         NaN         NaN                NaN               NaN                 NaN                     NaN                        NaN
 outlier_1       1   50     확실 이상        경계                92.229       0.309      -0.436              0.421              True                True                    True                       True
 outlier_2       2   47     확실 이상        경계               100.924       0.641      -0.357              0.621              True                True                    True                       True
 outlier_3       3   31     정상 유사        경계                73.791       0.291       0.184              0.029             False               False                   False                      False
 outlier_4

## 6. 그림

In [8]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sn = S[S.label == 0]
for s_, name, col in [(0, "정상 고부하(state 0)", "tab:blue"), (1, "정상 저부하(state 1)", "tab:green")]:
    q = sn[sn.state == s_]; ax.scatter(q.AI2_Current_rms, q.AI0_Vibration_rms, s=10, alpha=.45, c=col, label=f"{name} n={len(q)}")
so = S[S.label == 1]; ax.scatter(so.AI2_Current_rms, so.AI0_Vibration_rms, s=45, c="tab:red", marker="x", label=f"이상 n={len(so)}")
for _, r in so[so.seg_id.isin([3, 19, 20])].iterrows():
    ax.annotate(str(int(r.seg_id)), (r.AI2_Current_rms, r.AI0_Vibration_rms), textcoords="offset points", xytext=(5, 4), fontsize=9, color="tab:red")
for x_ in (95, 140): ax.axvline(x_, color="gray", ls="--", lw=.8)
ax.set_xlabel("세그먼트 전류 RMS (원시, DC 포함)"); ax.set_ylabel("세그먼트 AI0 진동 RMS"); ax.set_title("세그먼트 전류 RMS vs AI0 RMS (점선: 95 / 140)")
ax.legend(fontsize=8); paths.save_fig(FIG, "scatter_cur_vs_ai0_rms.png"); plt.close(fig)

km_ = M[(M.split == "group_kfold_seg") & (M.fold == "mean")]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
wd = 0.2
for j, (metric, ttl) in enumerate([("auc", "kfold 평균 AUC"), ("fpr_segment", "kfold 평균 fpr_segment")]):
    for i, m in enumerate(MODELS):
        v = [km_[(km_.model == m) & (km_.win_s == ws)][metric].iloc[0] for ws in (1.0, 2.0, 3.0)]
        axes[j].bar(np.arange(3) + (i - 1.5) * wd, v, wd, label=m)
    axes[j].set_xticks(range(3)); axes[j].set_xticklabels(["1초", "2초", "3초"]); axes[j].set_title(ttl)
axes[0].set_ylim(0.5, 1.0); axes[0].legend(fontsize=7); paths.save_fig(FIG, "rule_kfold_auc_fpr_segment.png"); plt.close(fig)
print(sorted(p.name for p in FIG.iterdir()))

saved figures\21_model_rule_baseline_CHS\scatter_cur_vs_ai0_rms.png
saved figures\21_model_rule_baseline_CHS\rule_kfold_auc_fpr_segment.png
['rule_kfold_auc_fpr_segment.png', 'scatter_cur_vs_ai0_rms.png']


## 7. 결론

**규칙별 비교 (1초, group_kfold_seg 평균 / time_block 평균)**

| 규칙 | kfold AUC | kfold fpr_segment | kfold n_detected(평균) | block AUC | block fpr_segment |
|---|---|---|---|---|---|
| rule1_rms_ai0 | 0.7864 | 0.0395 | 2.8 / 3.4 | 0.7786 | 0.0384 |
| rule2_rms_state | 0.7996 | 0.0468 | 2.8 / 3.4 | 0.7884 | 0.0456 |
| rule3_rms_state_dir | **0.8696** | **0.0358** | 2.8 / 3.4 | **0.8686** | 0.0187 |
| rule3b_rms_state_ratio | 0.8058 | 0.0399 | 2.8 / 3.4 | 0.7900 | 0.0196 |

- 2·3초 윈도우에서도 rule3가 최고다. kfold AUC는 rule1 / rule2 / rule3 / rule3b 순으로 2초 0.8581 / 0.8655 / 0.9100 / 0.8661, 3초 0.9030 / 0.9110 / 0.9496 / 0.8956이며, 3초에서는 rule3b가 rule1보다 낮다. rule1 1초는 11 노트북 smoke와 모든 fold에서 일치한다(4.1).
- **크기 규칙 안에서 상태별 임계(rule2)는 거의 이득이 없다**: AUC +0.013, 오히려 fpr_segment가 0.0395 → 0.0468로 늘었다. 상태 분리는 임계 정규화보다 방향 지표와 결합할 때 의미가 있다.
- **크기×방향(rule3)이 가장 낫다**: AUC +0.083(1초 kfold), fpr_segment 최저 수준. 단 `vib_corr01`은 세그먼트 브로드캐스트 값이라 윈도우 내용이 아닌 세그먼트 전체 성질이며, 실시간 경보에서는 세그먼트 종료 후에야 확정된다. 이 이득을 윈도우 단위 탐지 성능으로 그대로 읽으면 과대평가다.
- rule3b(윈도우 단위 AI0/AI1 비율)는 rule2 대비 AUC +0.006에 그쳤다. 비율 자체가 상태에 크게 의존(저부하 0.89 vs 고부하 0.48)하는데 상태별 기준화 없이 곱했기 때문이다.
- **탐지 수는 4개 규칙이 모두 같다**: 1초 윈도우가 있는 이상 세그먼트 17개 중 14개 탐지, 미탐은 3·19·20으로 동일(4개 규칙의 OOF 결과). 방향 결합은 순위(AUC)와 정상 오경보를 개선할 뿐 FN을 줄이지 못한다. 1초 윈도우가 없는 0·5·9·10(4개)은 판정 불가. 탐지 지연 중앙값은 규칙 무관 0.95초(kfold) / 체감 약 8.9초.

**FN 이원화**

- **세그먼트 3(진동·전류 모두 조용)**: `vib_grade` 정상 유사, `cur_grade` 경계, AI0 rms 중앙값 0.029로 정상 하위 수준, AC RMS 73.8은 정상 범위 안. 진동·전류 어느 채널에서도 신호가 약해 라벨 자체가 노이즈일 가능성이 있다.
- **세그먼트 19·20(진동 조용, 전류는 원시값만 확실)**: AI0 rms 0.026 / 0.023. 전류 원시 RMS 207/183의 91.7% / 96.4%는 DC이고 AC RMS는 59.6 / 35.2로 정상보다 낮다(V2-a). 즉 "전류가 확실한 이상"은 DC 오프셋 기준이며 표준 전처리에서는 관측되지 않는다. 진동 규칙의 FN은 이 세그먼트들에서 라벨 노이즈·채널 불일치를 먼저 의심하고, 전류 AC 하한(양측) 검정은 후속 검토로 남긴다.

**상태 정의**: 윈도우 AC RMS 2-means 경계 125.2로 저/고 2상태를 입력 가능한 값만으로 정의할 수 있고 세그먼트 k-means와 94.4% 일치(V2-b). 95/140 3구간은 95~140 완충 구간이 절반씩 섞여 있어 2상태로 단순화 가능.

**한계**
- 이상 이벤트 1건(세그먼트 21개, 1초 평가 가능 17개)이라 fold당 이상 세그먼트가 2~5개 수준이며 AUC·탐지 수 차이는 통계적으로 확정할 수 없다.
- `vib_corr01`은 세그먼트 브로드캐스트, 이상 세그먼트가 짧을수록(윈도우 수가 적을수록) 값이 불안정하다.
- 진동 게인 동일성 미확인(03 A-2), 정상은 한 날짜라 날짜 효과는 분리되지 않는다.
- 상태 경계와 상태별 임계는 fold마다 train 정상으로 새로 fit했으나 정상 파일이 한 날짜 77분이라 운전 상태 이동(time_block 마지막 블록의 상태 비율 변화, kickoff §4)에 대한 일반화는 별도 검증이 필요하다.